**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/10-nicolai-get-images-and-calibrate.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/watch-and-learn) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6J · Get Images → Calibrate → Undistort (Nicolai Nielsen)

_Lecture 6, video lab · Nicolai Nielsen._

> 🎬 **Watch first:** [Camera Calibration in less than 5 Minutes with OpenCV](https://www.youtube.com/watch?v=_-BTKiamRTg) (Nicolai Nielsen).
> Step-by-step notes for this video: [Tutorial 08 · Calibration in < 5 minutes](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#nicolai-5-minutes).

<a href="https://www.youtube.com/watch?v=_-BTKiamRTg"><img src="https://img.youtube.com/vi/_-BTKiamRTg/mqdefault.jpg" width="320" alt="Camera Calibration in less than 5 Minutes with OpenCV"></a>

> 🎬 **Watch first:** [Learn Camera Calibration in OpenCV with Python Script](https://www.youtube.com/watch?v=3h7wgR5fYik) (Nicolai Nielsen).
> Step-by-step notes for this video: [Tutorial 08 · The script, line by line](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/08_camera_calibration_videos.html#nicolai-script).

<a href="https://www.youtube.com/watch?v=3h7wgR5fYik"><img src="https://img.youtube.com/vi/3h7wgR5fYik/mqdefault.jpg" width="320" alt="Learn Camera Calibration in OpenCV with Python Script"></a>

## What the videos do, in plain words

* **Script 1, `get_images.py`:** open the webcam; press **s** to save a picture of the chessboard, **ESC** to stop.
  10–20 pictures at different positions and angles are enough. (Our copy is in this folder.)
* **Script 2, `calibration.py`:** for every picture find the corners, collect **object points** (where the corners are on the
  flat board) and **image points** (where they are in the picture), call `calibrateCamera`, **save** the camera matrix and
  distortion with `pickle`, then **undistort** in two ways (`undistort` and `remap`) and print a **total error**.
* Zhang's paper (shown in the video) says a handful of views (3–5) already gives a small error. More views mainly add
  robustness.

We use the 20 dash-cam pictures from Lab 6D instead of a webcam, so this runs on Colab. Run `get_images.py` at home to make
your own set.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py', 'get_images.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('road_cal')
print("data ready")

In [ ]:
import glob, json, os, pickle, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=4, suppress=True)
print("OpenCV", cv2.__version__)

## 1. The settings (top of Nicolai's script)

`chessboardSize` = inner corners (his board was 24 × 17; ours is **9 × 6**), `frameSize` = (width, height) of the images.

In [ ]:
chessboardSize = (9, 6)
frameSize = (1280, 720)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)   # for cornerSubPix

# object points (0,0,0), (1,0,0) ... (8,5,0), as in the video
objp = np.zeros((chessboardSize[0] * chessboardSize[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboardSize[0], 0:chessboardSize[1]].T.reshape(-1, 2)
size_of_chessboard_squares_mm = 20
objp = objp * size_of_chessboard_squares_mm

objpoints = []   # 3-D points in the real world
imgpoints = []   # 2-D points in the image plane
images = sorted(glob.glob("data/camera_cal/*.jpg"))
print(len(images), "images")

## 2. Find the corners in every image

Nicolai's loop, plus one line he does not need with his own webcam: **two of these images are 1281 × 721**, so we resize
every image to `frameSize` (otherwise `calibrateCamera` fails with a size error).

In [ ]:
for image in images:
    img = cv2.imread(image)
    if img.shape[1::-1] != frameSize:
        img = cv2.resize(img, frameSize)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, chessboardSize, None)
    if ret:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners.reshape(-1, 1, 2), (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)
print(f"corners found in {len(imgpoints)} of {len(images)} images")

## 3. Calibrate and save with pickle (three files, as in the video)

In [ ]:
ret, cameraMatrix, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, frameSize, None, None)
print("RMS:", ret); print(cameraMatrix); print(dist)

pickle.dump((cameraMatrix, dist), open("calibration.pkl", "wb"))
pickle.dump(cameraMatrix, open("cameraMatrix.pkl", "wb"))
pickle.dump(dist, open("dist.pkl", "wb"))
cameraMatrix2, dist2 = pickle.load(open("calibration.pkl", "rb"))       # load them back in your own project
print("reloaded OK:", np.allclose(cameraMatrix2, cameraMatrix))

## 4. Undistort two ways: `undistort` vs `remap`

Both give the same image; `remap` is faster for **video** because the map is computed **once**.

In [ ]:
img = cv2.imread("data/camera_cal/calibration1.jpg")
h, w = img.shape[:2]
newCameraMatrix, roi = cv2.getOptimalNewCameraMatrix(cameraMatrix, dist, (w, h), 1, (w, h))

t0 = time.perf_counter()
dst1 = cv2.undistort(img, cameraMatrix, dist, None, newCameraMatrix)
t1 = time.perf_counter()
mapx, mapy = cv2.initUndistortRectifyMap(cameraMatrix, dist, None, newCameraMatrix, (w, h), 5)
t2 = time.perf_counter()
dst2 = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
t3 = time.perf_counter()
print(f"undistort {1000 * (t1 - t0):.1f} ms | build map once {1000 * (t2 - t1):.1f} ms | remap {1000 * (t3 - t2):.1f} ms per frame")

x, y, rw, rh = roi
fig, ax = plt.subplots(1, 3, figsize=(22, 4.5))
ax[0].imshow(img[:, :, ::-1]); ax[0].set_title("original")
ax[1].imshow(dst1[y:y + rh, x:x + rw, ::-1]); ax[1].set_title("undistort + crop to roi")
ax[2].imshow(dst2[y:y + rh, x:x + rw, ::-1]); ax[2].set_title("remap + crop to roi")
for a in ax: a.axis("off")
plt.show()

## 5. Nicolai's "total error" vs the RMS

His script prints

```python
mean_error = 0
for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], cameraMatrix, dist)
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    mean_error += error
print("total error:", mean_error / len(objpoints))
```

Compute it below and compare it with `ret` (the RMS that `calibrateCamera` returns).

In [ ]:
total_error = 0.0   # TODO: implement Nicolai's loop

print("total error:", total_error, "   RMS:", ret)

In [ ]:
C.check_total_error(total_error, ret)

**Why are they different?** `cv2.norm(..., NORM_L2)` is √(Σ d²) over *all* corners of a view; dividing by N (not √N)
makes it smaller than the per-corner RMS by a factor √N. Neither is wrong, but they are **not comparable**: when someone says
"my error is 0.03", ask which formula. In this course we report the **RMS in pixels**.

## 6. How many views do you need? (the Zhang plot from the video)

In [ ]:
rng = np.random.default_rng(0)
counts, fx_spread = [3, 5, 8, 12, len(objpoints)], []
for n in counts:
    fx = []
    for _ in range(8):
        idx = rng.choice(len(objpoints), n, replace=False)
        fx.append(cv2.calibrateCamera([objpoints[i] for i in idx], [imgpoints[i] for i in idx], frameSize, None, None)[1][0, 0])
    fx_spread.append(np.std(fx))
plt.figure(figsize=(8, 4)); plt.plot(counts, fx_spread, "o-"); plt.xlabel("number of views"); plt.ylabel("std of fx over random subsets (px)")
plt.title("more views -> more stable result"); plt.grid(alpha=0.3); plt.show()

## Your turn

1. Run `get_images.py` with your laptop webcam (print or show `pattern.png`), take 15 pictures, and run sections 1–5 on
   **your** images (change `images`, `chessboardSize`, `frameSize`). Report the RMS.
2. Change `size_of_chessboard_squares_mm` from 20 to 100. Which outputs change (K? dist? tvecs?) and why?
3. Section 6: why can 3–5 views be "enough" in Zhang's paper but not enough in practice?